<a href="https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import pandas as pd
import numpy as np
import os
import json

# Load the starter dataset
possible_paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv", 
    "../../data/raw/content_refresh_anonymized.csv"
]

data_loaded = False
for path in possible_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"Data loaded from: {path}")
        print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
        data_loaded = True
        break

if not data_loaded:
    raise FileNotFoundError("Could not find content_refresh_anonymized.csv in expected locations")

# Create the target label
df['is_declining'] = (df["trend_direction"].str.lower() == "down").astype(int)
print(f"Target declining rate: {df['is_declining'].mean():.3f}")

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked Actions and Strategic Reason Codes
For the Refresh / Content Opportunity Scoring lane, I transform model predictions into actionable content team directives using clear reason codes:

**Action Categories**:
- **CONTENT_REFRESH_PRIORITY** (Code: DECLINING_HIGH_VISIBILITY): Pages showing decline patterns with good search visibility - these are the top priority for content review
- **CONTENT_REFRESH_MODERATE** (Code: STALE_VISIBLE): Old content (≥180 days) that still gets impressions - refresh candidates with moderate priority  
- **MONITOR_STABLE** (Code: STABLE_PERFORMING): Pages with healthy or growing trends - no immediate action needed
- **REVIEW_THRESHOLD** (Code: LOW_VISIBILITY): Pages with very low impressions - not worth refresh effort without addressing visibility first

**Reason Code Logic**:
- **DECLINING_HIGH_VISIBILITY**: trend_direction='down' AND impressions_90d ≥ 100 AND avg_position ≤ 20
- **STALE_VISIBLE**: days_since_last_update ≥ 180 AND impressions_90d ≥ 500 AND trend_direction != 'down'
- **STABLE_PERFORMING**: trend_direction IN ['up', 'stable'] AND impressions_90d ≥ 50
- **LOW_VISIBILITY**: impressions_90d < 50 (regardless of other signals)

In [ ]:
# Apply the action playbook logic to create ranked queue with reason codes
conditions = [
    (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100) & (df["avg_position"] <= 20),
    (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500) & (df["trend_direction"] != "down"),
    (df["trend_direction"].isin(["up", "stable"])) & (df["impressions_90d"] >= 50),
    (df["impressions_90d"] < 50)
]

choices_action = ["CONTENT_REFRESH_PRIORITY", "CONTENT_REFRESH_MODERATE", "MONITOR_STABLE", "REVIEW_THRESHOLD"]
choices_reason = ["DECLINING_HIGH_VISIBILITY", "STALE_VISIBLE", "STABLE_PERFORMING", "LOW_VISIBILITY"]

df["action_label"] = np.select(conditions, choices_action, default="MONITOR_STABLE")
df["reason_code"] = np.select(conditions, choices_reason, default="STABLE_PERFORMING")

# Create priority score for ranking (highest priority gets highest score)
priority_scores = {
    "CONTENT_REFRESH_PRIORITY": 100,
    "CONTENT_REFRESH_MODERATE": 60,
    "MONITOR_STABLE": 10,
    "REVIEW_THRESHOLD": 5
}
df["priority_score"] = df["action_label"].map(priority_scores)

# Secondary ranking by impressions within priority tier
df["final_score"] = df["priority_score"] + (df["impressions_90d"] / df["impressions_90d"].max() * 10)

# Create the final ranked queue
final_ranked_queue = df.sort_values(by="final_score", ascending=False)

print("=== ACTION PLAYBOOK COHORT DISTRIBUTION ===")
print(final_ranked_queue["action_label"].value_counts().to_string())
print(f"\nTotal pages in ranked queue: {len(final_ranked_queue):,}")

# Show top 10 with their reason codes
print("\n=== TOP 10 PRIORITY PAGES ===")
top_10 = final_ranked_queue[['action_label', 'reason_code', 'impressions_90d', 'avg_position', 'trend_direction']].head(10)
print(top_10.to_string(index=False))

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use Framework and Boundary Limits

**Intended Domain Use**: This content action playbook serves as a decision-support tool for content teams and SEO specialists to prioritize refresh efforts. It helps identify which pages showing decline patterns are most worth reviewing first, based on observed 90-day search performance signals.

**Who Uses This**: 
- Content editors and SEO specialists who decide where to invest editorial time
- Content strategists planning quarterly refresh calendars
- Site managers overseeing large content portfolios

**Known System Limits**:
- **No causal claims**: The playbook identifies observed decline patterns, not guaranteed refresh outcomes
- **Algorithm shifts**: Cannot predict Google algorithm updates or ranking changes
- **External factors**: Seasonal trends, competitor actions, and market changes not captured
- **Current-state proxy**: Uses trend_direction as a proxy label, not true future performance
- **Single-channel data**: Only considers organic search, not other traffic sources

**Where It Stops Being Valid**:
- For brand new pages (< 90 days old) - insufficient history
- For pages with zero impressions - no measurable performance to assess
- During major search algorithm updates - historical patterns may not apply
- For highly seasonal content - normal seasonal fluctuations may be misidentified as decline

In [ ]:
print(f"Action playbook boundaries defined for {len(final_ranked_queue):,} pages")
print(f"Primary users: Content teams, SEO specialists, content strategists")

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Guardrails and the Automation No-Go List

**Manual Review Requirements**:
Before taking action on flagged pages, content teams must manually verify:

1. **Seasonal pattern check**: Confirm decline isn't due to normal seasonal fluctuations (e.g., holiday content, annual events)
2. **Brand criticality assessment**: Ensure pages aren't core brand assets or homepage that require different treatment
3. **Technical audit**: Verify there are no technical SEO issues (crawl errors, indexing problems) causing the decline
4. **Content intent review**: Confirm the page's primary purpose and target audience are still relevant

**Automation No-Go List** (must always have human review):
- **Core brand pages**: Homepage, about pages, core product pages - never automate refresh decisions
- **Legal/compliance content**: Privacy policy, terms of service, disclaimers - require legal review
- **Financial/medical content**: YMYL (Your Money Your Life) pages - requires subject matter expert review
- **Active campaign landing pages**: Pages tied to current paid campaigns - may reflect temporary performance
- **New product launches**: Pages less than 90 days old - insufficient performance history

**Human Review Checklist**:
- [ ] Confirm not a core brand page
- [ ] Verify decline isn't seasonal/temporary
- [ ] Check for technical SEO issues
- [ ] Assess content relevance and accuracy
- [ ] Consider business impact of changes

In [ ]:
# Count pages that would require human review based on no-go criteria
# Estimate pages that might be core brand or critical content
# (In real implementation, this would use actual page type classification)

estimated_no_go_pages = final_ranked_queue[
    (final_ranked_queue['content_age_days'] < 90) |  # New pages
    (final_ranked_queue['impressions_90d'] > final_ranked_queue['impressions_90d'].quantile(0.95))  # Top visibility pages (likely core)
].shape[0]

print(f"Estimated pages requiring human review (no-go criteria): {estimated_no_go_pages:,}")
print(f"This represents ~{estimated_no_go_pages/len(final_ranked_queue)*100:.1f}% of total pages")
print("Human review guardrails protect critical brand and compliance content from automated decisions")

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Operational Monitoring and Retraining Triggers

To ensure the action playbook remains effective over time, I need to monitor for data drift and performance degradation. The following triggers indicate when the model should be retrained or the playbook recalibrated:

**Data Drift Triggers**:
- **Feature distribution shift**: If the distribution of key features (impressions_90d, avg_position, ctr) shifts significantly from the baseline (±15% change in median values)
- **Target class imbalance**: If the declining rate changes dramatically (±10 percentage points from the baseline ~54%)
- **New content types**: If new content_type categories appear that weren't in the training data

**Performance Degradation Triggers**:
- **Precision@50 drop**: If the measured Precision@50 on new data falls below 80% of the baseline performance
- **False positive increase**: If the false positive rate increases above 30% (indicating too many stable pages flagged)
- **Reason code distribution shift**: If the distribution of reason codes changes significantly from the baseline

**Manual Review Triggers**:
- **Human feedback loop**: If content teams report that >25% of flagged pages don't actually need refresh
- **Business outcome tracking**: If refresh actions on top-priority pages don't show measurable improvement after 30 days

**Retraining Process**:
1. Collect new 90-day performance data
2. Re-run the entire pipeline with updated data
3. Re-validate using client-holdout split
4. Compare new model performance against baseline
5. Update action thresholds if performance gap narrows

In [ ]:
# Calculate baseline metrics for monitoring
baseline_metrics = {
    "baseline_impressions_median": float(df["impressions_90d"].median()),
    "baseline_avg_position_median": float(df["avg_position"].median()),
    "baseline_ctr_median": float(df["ctr"].median()),
    "baseline_declining_rate": float(df["is_declining"].mean()),
    "precision_50_threshold": 0.8,  # 80% of baseline performance
    "drift_tolerance": 0.15,  # 15% change threshold
    "fp_rate_threshold": 0.30  # 30% false positive rate threshold
}

print("=== MONITORING BASELINE METRICS ===")
for key, value in baseline_metrics.items():
    print(f"{key}: {value}")

# Save metrics for future comparison
os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/action_playbook_baseline_metrics.json", "w") as f:
    json.dump(baseline_metrics, f, indent=2)

print("\nBaseline metrics saved to work/outputs/action_playbook_baseline_metrics.json")

### Cost/Value Analysis: Playbook ROI vs Random Selection

**Random Selection Baseline**:
- Randomly picking pages for refresh would achieve ~54% accuracy (the base declining rate at 0.542)
- Content teams would waste time on many stable pages that don't need attention
- High opportunity cost: editorial time spent on wrong pages

**Playbook Advantages**:
- **Precision improvement**: The hand-written baseline achieves 0.960 Precision@50, while the Random Forest model achieves 0.780 Precision@50 (1.44× better than random selection at 0.542)
- **Editorial efficiency**: Content teams focus on pages most likely to actually need refresh
- **Priority ranking**: Reason codes explain WHY each page is flagged, helping teams understand the issues

**Quantified ROI Estimate**:
- **Time savings**: If a content editor spends 4 hours per page refresh, focusing on the top 50 pages with 78% precision vs 54% random = 12 extra pages correctly identified = 48 hours saved per refresh cycle
- **Traffic impact**: Prioritizing declining pages with visibility means refresh efforts are focused where they can recover the most organic traffic
- **Team productivity**: Less time debating which pages to refresh, more time executing refreshes

**Cost Considerations**:
- **False positive cost**: Wasted editorial time on pages that don't actually need refresh (~22% of top 50 with model vs ~46% with random)
- **False negative cost**: Missing declining pages that could have been recovered (model finds 78% vs 54% random)
- **Monitoring overhead**: Time required for human review guardrails and monitoring triggers

**Net Value**: The playbook provides positive ROI when editorial time is more valuable than the opportunity cost of missed recoveries, which is typically true for teams managing large content portfolios.

### Cost/Value Analysis: Playbook ROI vs Random Selection

**Random Selection Baseline**:
- Randomly picking pages for refresh would achieve ~54% accuracy (the base declining rate)
- Content teams would waste time on many stable pages that don't need attention
- High opportunity cost: editorial time spent on wrong pages

**Playbook Advantages**:
- **Precision improvement**: Based on Week-5 results, the model achieves ~3x better Precision@50 than random (0.740 vs 0.240 baseline)
- **Editorial efficiency**: Content teams focus on pages most likely to actually need refresh
- **Priority ranking**: Reason codes explain WHY each page is flagged, helping teams understand the issues

**Quantified ROI Estimate**:
- **Time savings**: If a content editor spends 4 hours per page refresh, focusing on the top 50 pages with 74% precision vs 24% random = 25 extra pages correctly identified = 100 hours saved per refresh cycle
- **Traffic impact**: Prioritizing declining pages with visibility means refresh efforts are focused where they can recover the most organic traffic
- **Team productivity**: Less time debating which pages to refresh, more time executing refreshes

**Cost Considerations**:
- **False positive cost**: Wasted editorial time on pages that don't actually need refresh (~26% of top 50 with model vs ~76% with random)
- **False negative cost**: Missing declining pages that could have been recovered (model finds 74% vs 24% random)
- **Monitoring overhead**: Time required for human review guardrails and monitoring triggers

**Net Value**: The playbook provides positive ROI when editorial time is more valuable than the opportunity cost of missed recoveries, which is typically true for teams managing large content portfolios.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Ranked actions have clear reason codes explaining WHY each page is flagged
- [ ] Intended use and limits are clearly documented
- [ ] Human review guardrails and no-go list protect critical content
- [ ] Monitoring triggers and retraining process are defined
- [ ] Cost/value analysis shows ROI vs random selection
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### 5. Final Workspace Asset Exports
This section saves our analytical deliverables into the project's local outputs folder. These data receipts act as the verified foundation for the deployed numbers and data frames that our final research report builds upon next week.


In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import json

# 1. Force reset workspace to default root if the environment path is completely lost
try:
    current_dir = os.getcwd()
except FileNotFoundError:
    print("[PATH ALERT] Current working directory lost due to folder cleanup. Resetting to /content...")
    os.chdir("/content")
    current_dir = os.getcwd()

print(f"Current Execution Directory: {current_dir}")

# 2. Determine the exact path of your cloned workspace repository
repo_name = "flyrank-ml-assignments"
if repo_name in current_dir:
    base_target_dir = current_dir.split(repo_name)[0] + repo_name
else:
    base_target_dir = os.path.join("/content", repo_name)

# 3. Create absolute fallback paths if local workspace folders do not exist
outputs_dir = os.path.join(base_target_dir, "work/outputs")
metrics_dir = os.path.join(base_target_dir, "work/metrics")

os.makedirs(outputs_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

# 4. Export the primary actionable prioritizations queue dataframe frame
queue_destination = os.path.join(outputs_dir, "baseline_action_score.csv")
final_ranked_queue.to_csv(queue_destination, index=False)
print(f"[EXPORT PASS] Actionable queue matrix saved to: {queue_destination}")

# 5. Export structured monitoring telemetry profiling JSON receipt
metrics_destination = os.path.join(metrics_dir, "playbook_telemetry_receipt.json")
with open(metrics_destination, "w") as json_file:
    json.dump(rolling_stats_profile, json_file, indent=4)
print(f"[EXPORT PASS] Telemetry verification configuration receipt logged to: {metrics_destination}")


Current Execution Directory: /content
[EXPORT PASS] Actionable queue matrix saved to: /content/flyrank-ml-assignments/work/outputs/baseline_action_score.csv
[EXPORT PASS] Telemetry verification configuration receipt logged to: /content/flyrank-ml-assignments/work/metrics/playbook_telemetry_receipt.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.